# Заголовок Общий

## Заголовок EDA

Описание

### Импорты

In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from geopandas import GeoDataFrame
from shapely.geometry import Point
import seaborn as sns

from src.features.geo import water_fraction, extract_polygon, within_shape
from src.plotting.plots import plot_ct
from src.plotting.style import water_col, field_markers, field_lgnd
from src.preprocessing.preprocessing import construct_basin_location_mapper, \
    construct_basin_location_mapper_with_external, split_strings, cat_diff

plt.rcParams['axes.grid'] = False

### Загрузка и предобработка данных

#### Выборки

In [2]:
train_oil_df = pd.read_csv("../data/train_oil.csv")
test_oil_df = pd.read_csv("../data/oil_test.csv")

In [3]:
train_oil_df.columns = [c.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_") for c in
                        train_oil_df.columns]
train_oil_df.columns

Index(['field_name', 'reservoir_unit', 'country', 'region', 'basin_name',
       'tectonic_regime', 'latitude', 'longitude', 'operator_company',
       'onshore_offshore', 'hydrocarbon_type', 'reservoir_status',
       'structural_setting', 'depth', 'reservoir_period', 'lithology',
       'thickness_gross_average_ft', 'thickness_net_pay_average_ft',
       'porosity', 'permeability'],
      dtype='str')

In [4]:
test_oil_df.columns = [c.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_") for c in
                        test_oil_df.columns]

In [5]:
train_str_cols = train_oil_df.select_dtypes(include='str').columns
test_str_cols = test_oil_df.select_dtypes(include='str').columns
train_oil_df[train_str_cols] = train_oil_df[train_str_cols].apply(lambda x: x.str.lower())
test_oil_df[test_str_cols] = test_oil_df[test_str_cols].apply(lambda x: x.str.lower())

#### Геоданные
Основной источник геоданных - это https://www.naturalearthdata.com 

<img src="../doc/img/NEV-Logo-color.png" width="500">

(Атрибуция: Made with Natural Earth. Free vector and raster map data @ naturalearthdata.com)

Из этого источника мы возьмём ряд датасетов и поделим их по группам. Используемые масштабы:
- 10m - 1:10000000, 1см = 100км;
- 50m - 1:50000000, 1см = 500км;
- 110m - 1:110000000, 1см = 1100км.

Дополнительный источник данных относится к бассейнам: ArcGIS REST Services Directory "Layer: Basins classified by sub regime CGG GeoVerse (ID:0)": https://services5.arcgis.com/33PZ8RasWNSHnkUG/ArcGIS/rest/services/Sedimentary_Basins_of_the_World/FeatureServer/0. Данные были загружены и сохранены в /data/basins.csv.

##### Гео-датасеты

In [6]:
land = gpd.read_file("../data/ne_50m_land.zip")['geometry']
land_10m = gpd.read_file("../data/ne_10m_land.zip")['geometry']
ocean_50m = gpd.read_file("../data/ne_50m_ocean.zip")['geometry']
ocean_110m = gpd.read_file("../data/ne_110m_ocean.zip")['geometry']
ne_50m_geography_marine_polys = gpd.read_file("../data/ne_50m_geography_marine_polys.zip")
lakes = gpd.read_file("../data/ne_50m_lakes.zip")['geometry']
geography_regions = gpd.read_file("../data/ne_50m_geography_regions_polys.zip")
rivers_lake_centerlines = gpd.read_file("../data/ne_50m_rivers_lake_centerlines.zip")['geometry']
major_islands = gpd.read_file("../data/ne_50m_coastline.zip")['geometry']
minor_islands = gpd.read_file("../data/ne_10m_minor_islands.zip")['geometry']

##### Формирование подмножеств геопризнаков

In [7]:
bays = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'bay']['geometry']
gulfs = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'gulf']['geometry']
straits = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'strait']['geometry']
channels = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'channel']['geometry']
sounds = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'sound']['geometry']
rivers = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'river']['geometry']
isthmuses = geography_regions[geography_regions.FEATURECLA == 'Isthmus']['geometry']
deltas = geography_regions[geography_regions.FEATURECLA == 'Delta']['geometry']
coasts = geography_regions[geography_regions.FEATURECLA == 'Coast']['geometry']
lake_regions = geography_regions[geography_regions.FEATURECLA == 'Lake']['geometry']
islands = geography_regions[geography_regions.FEATURECLA.isin(['Island group', 'Island'])]['geometry']

In [8]:
bunyu_coords = Point(117.833, 3.5)
bunyu_island = gpd.GeoSeries(
    [extract_polygon(land_10m[land_10m.geometry.contains(bunyu_coords)].iloc[0], bunyu_coords)],
    crs="EPSG:4326"
)

In [9]:
all_lakes = gpd.GeoDataFrame(
    geometry=pd.concat([lakes, rivers_lake_centerlines, lake_regions], ignore_index=True),
    crs="EPSG:4326").dissolve()

In [10]:
all_islands = gpd.GeoDataFrame(
    geometry=pd.concat([islands, major_islands, minor_islands, bunyu_island], ignore_index=True),
    crs="EPSG:4326").dissolve()

##### Бассейны
Onshore/offshore mapping получен из датасета ARCGIS [Sedimentary_Basins_of_the_World](https://services5.arcgis.com/33PZ8RasWNSHnkUG/ArcGIS/rest/services/Sedimentary_Basins_of_the_World/FeatureServer/0) и преобразован в [load_basins_data](load_basins_data.ipynb).

In [11]:
basins = pd.read_csv("../data/basins.csv")
basins_mapped = pd.read_csv("../data/basins_mapped.csv")

In [12]:
basins.head()

,basin_name,location
0,Faroe - Shetland Escarpment,Offshore
1,Porcupine,Offshore
2,North Lewis,Offshore
3,South Celtic Sea,Offshore
4,Blake Plateau Ultradeep,Offshore


In [13]:
basins_mapped.head()

,basin_name,location
0,amu darya,onshore
1,anadarko,onshore
2,appalachian,onshore
3,aquitaine,onshore-offshore
4,arkoma,onshore


### Ознакомление с датасетом

In [14]:
train_oil_df.head(5)

,field_name,reservoir_unit,country,region,basin_name,tectonic_regime,latitude,longitude,operator_company,onshore_offshore,hydrocarbon_type,reservoir_status,structural_setting,depth,reservoir_period,lithology,thickness_gross_average_ft,thickness_net_pay_average_ft,porosity,permeability
0,zhirnov,melekeskian,russia,former soviet union,volga-ural,compression/evaporite,51.0000,44.8042,nizhnevolzhsknet,onshore,oil,declining production,foreland,1870,carboniferous,sandstone,262.0,33.0,24.0,30.0
1,lagoa parda,lagoa parda (urucutuca),brazil,latin america,espirito santo,extension,-19.6017,-39.8332,petrobras,onshore,oil,nearly depleted,passive margin,4843,paleogene,sandstone,2133.0,72.0,23.0,350.0
2,abqaiq,arab d,saudi arabia,middle east,the gulf,compression/evaporite,26.0800,49.8100,saudi aramco,onshore,oil,rejuvenating,foreland,6050,jurassic,limestone,250.0,184.0,21.0,410.0
3,murchison,brent,uk /norway,europe,north sea northern,extension,61.3833,1.7500,cnr,offshore,oil,nearly depleted,rift,8988,jurassic,sandstone,425.0,300.0,22.0,750.0
4,west pembina,nisku (pembina l pool),canada,north america,western canada,compression,53.2287,-115.8008,numerous,onshore,oil,unknown,foreland,9306,devonian,dolomite,233.0,167.0,11.8,1407.0


**Идентификаторы месторождения**
* Field name - название месторождения
* Reservoir unit - юнит месторождения

Название и юнит месторождения, как мы убедимся, уникально идентифицируют месторождение. Кроме того, названия содержат полезную для задачи информацию. 

**Локация месторождения**
* Country - страна расположения
* Region - регион расположения
* Latitude - широта
* Longitude - долгота

Координаты являются очень важным источником сведений о месторждении. Многие признаки и анализ будут выполнены, как раз, на координатах. Страна и регион, сами по себе несут ограниченную полезность.

**Геологические характеристики**
* Basin name - название бассейна пород;
* Reservoir period - литологический период;
* Lithology (main) - литология;
* Tectonic regime - тектонический режим;
* Structural setting - структурные свойства.

Предполагается, что литология, тектоника и структурные характеристики помогут отличать береговые и оффшорные месторождения. Бассейн будем рассматривать, как агрегатор периода и других геологических признаков.

**Эксплуатационные характеристики**
* Operator company - название компании. В целом, ожидается ограниченная полезность этого показателя, т.к. под месторождения могут создаваться отдельные юрлица, чтобы ограничить ответственность и проблемы владельца одим ЮЛ, упростить получение лицензий и компартментализировать эксплуатацию основных средств. Некоторые операторы, вероятно, могут специализироваться на береговых/оффшорных месторождений. Имеет смысл проверить.
* Hydrocarbon type (main) - тип углеводорода. Признак сообщает, скорее преимущественный углеводород, который добывается на месторождении, хотя на нефтяных скважинах добывается и попутный газ. 
* Reservoir status (current) - статус месторождения. Скорее всего, признак будет бесполезен, т.к. это эксплуатационное состояние, которое, в принципе, не может сообщать о локации.
* Depth (top reservoir ft TVD) - глубина. 
* Thickness (gross average ft) - общая толщина
* Thickness (net pay average ft) - эффективная толщина
* Porosity (matrix average 20) - Permeability (air average mD) – проницаемость

Толщина и проницаемость, вероятно, могут содержать какую-то информацию о локации.

В качестве глубины указана [True Vertical Depth](https://en.wikipedia.org/wiki/True_vertical_depth) - длина воображаемой вертикальной линии от поверхности по координатам до точки интереса (`a` на иллюстрации). Т.к. месторождения классифицируются по глубинам залегания, а по координатам можно определить находится ли там вода или суша, то можно было бы построить из этих сведений хорошие признаки. Существуют датасеты с бариметрическими данными для координат (например, [GEBCO](https://www.gebco.net/data-products/gridded-bathymetry-data)), по которым можно было бы определить глубину от поверхности до дна и вычислить TVD (True Vertical Depth Sub-Sea). Но сам датасет достаточно тяжёлый, поэтому рассмотрим его использование, если нужно будет повысить точность.

<img src="../doc/img/TVD.png" width="200">

Источник: https://en.wikipedia.org/wiki/True_vertical_depth

Целевая переменная:
* Onshore or offshore - целевая переменная (ONSHORE - 1, OFFSHORE - 0, ONSHORE-OFFSHORE - 2)

Здесь важно отметить, что под береговыми ("ONSHORE") месторождениями понимаются, в том числе, и те, которые находятся в озёрах. Это мы выясним дополнительно в ходе анализа.  


### Уникальность
Уникальность задана двумя полями.

In [15]:
unique_fields = train_oil_df[['field_name', 'reservoir_unit']].duplicated(keep=False)
train_oil_df[unique_fields].sort_values(by=['field_name'])

,field_name,reservoir_unit,country,region,basin_name,tectonic_regime,latitude,longitude,operator_company,onshore_offshore,hydrocarbon_type,reservoir_status,structural_setting,depth,reservoir_period,lithology,thickness_gross_average_ft,thickness_net_pay_average_ft,porosity,permeability


### Пропущенные значения

In [16]:
missing_val_cnts = train_oil_df.isna().sum()
missing_val_cnts[missing_val_cnts > 0]

country       27
region        38
basin_name    38
latitude      27
longitude     30
dtype: int64

Заметим:
- В колонках определяющих уникальность, пропусков нет. Значит другие значения можно заполнить по этим колонкам.
- Количество отсутствующих значений для региона и названия бассейна одинаковое. Проверим, что значения отсутствуют в одних и тех же записях.

In [17]:
(train_oil_df['region'].isna() & train_oil_df['basin_name'].isna()).sum()

np.int64(38)

Заполним регион и страну значением 'unknown'. Бассейны рассмотрим вместе с координатами.

In [18]:
train_oil_df['region'] = train_oil_df['region'].fillna('unknown')
train_oil_df['country'] = train_oil_df['country'].fillna('unknown')

Посмотрим, как отсутствуют координаты.

In [19]:
missing_lat = train_oil_df['latitude'].isna()
missing_lon = train_oil_df['longitude'].isna()

In [20]:
missing_coords_by_field = train_oil_df[missing_lat | missing_lon][
    ['field_name', 'reservoir_unit', 'country']]
missing_coords_by_field

,field_name,reservoir_unit,country
11,badr el din-2,bahariya,unknown
12,bridger lake,dakota sandstone (lower member),unknown
15,scott,sgiath-piper,unknown
28,zakum,thamama zone ii,unknown
33,gasikule,upper ganchaigou-lower youshas,unknown
35,uzen,units xiii-xviii,unknown
37,ula,ula,unknown
49,yibal,khuff,unknown
50,palm valley,pacoota and stairway,unknown
55,wubaiti,huanglong,unknown


Важно отметить, что названия и юниты месторождения нам даны даже там, где координат нет, а значит, можно восстановить их.

In [21]:
test_oil_df.isna().sum()

field_name                       0
reservoir_unit                   0
country                         13
region                          16
basin_name                       8
tectonic_regime                  0
latitude                        13
longitude                       16
operator_company                 0
hydrocarbon_type                 0
reservoir_status                 0
structural_setting               0
depth                            0
reservoir_period                 0
lithology                        0
thickness_gross_average_ft       0
thickness_net_pay_average_ft     0
porosity                         0
permeability                     0
dtype: int64

### Заполнение пропущенных координат

Т.к. названия и юниты месторождений нам даны и их всего 30, я решил их восстановить. Геомэппинг по названиям не дал надёжных результатов. Пришлось искать через поисковые системы. Оказалось, что задача непростая, т.к. для некоторых месторождений сведеий почти нет (Yakin Complex, например, находится в районе, который не указан в датасете), а месторождения в США имеют свою систему поиска.

In [ ]:
missing_coordinates = pd.read_csv("../data/missing_coordinates.csv")
missing_coordinates = missing_coordinates.fillna("")
display(HTML(missing_coordinates.to_html()))

In [ ]:
train_oil_df_filled = train_oil_df.merge(
    missing_coordinates[["field_name", "reservoir_unit", "latitude", "longitude"]],
    on=["field_name", "reservoir_unit"],
    how='left', suffixes=('', '_fill'))
train_oil_df_filled['latitude'] = train_oil_df_filled['latitude'].fillna(train_oil_df_filled['latitude_fill'])
train_oil_df_filled['longitude'] = train_oil_df_filled['longitude'].fillna(train_oil_df_filled['longitude_fill'])
train_oil_df_filled.drop(columns=['latitude_fill', 'longitude_fill'], inplace=True)
train_oil_df_filled[['latitude', 'longitude']].isna().sum()

### Заполнение бассейнов
Т.к. названия и юниты месторождений нам даны и их всего 38, я аналогично решил восстановить. Пользовался теми же источниками, что и для координат, где названия бассейновы были доступны, где не было таких сведений, пользовался DeepSeek'ом. Сверялся с [Sedimentary_Basins_of_the_World](https://services5.arcgis.com/33PZ8RasWNSHnkUG/ArcGIS/rest/services/Sedimentary_Basins_of_the_World/FeatureServer/0).

In [ ]:
missing_basins = pd.read_csv("../data/missing_basins.csv")

In [ ]:
train_oil_df_filled = train_oil_df_filled.merge(missing_basins[["basin_name", "field_name", "reservoir_unit"]],
                                                on=["field_name", "reservoir_unit"],
                                                how='left', suffixes=('', '_fill'))
train_oil_df_filled['basin_name'] = train_oil_df_filled['basin_name'].fillna(train_oil_df_filled['basin_name_fill'])
train_oil_df_filled.drop(columns=['basin_name_fill'], inplace=True)
train_oil_df_filled[['basin_name']].isna().sum()

### Соотношение классов

In [ ]:
train_oil_df['onshore_offshore'].value_counts()

Больше всего в датасете береговых месторождений, присутствует лишь 5 смешанных.

### Совместное использование бассейнов странами

In [ ]:
basin_country_shared = train_oil_df.groupby('basin_name', as_index=False)['country'].unique()
basin_country_shared['shared_by'] = basin_country_shared['country'].apply(lambda x: len(x))
basin_country_shared = basin_country_shared[basin_country_shared['shared_by'] > 1].sort_values(by='shared_by',
                                                                                               ascending=False)
basin_country_shared

Важно отметить следующее:
- Есть бассейны, которые используются несколькими странами;
- есть дубликаты вида uk/norway, norway/uk, а значит можно их унифицировать и расширить обучающую выборку;

In [ ]:
uk_nor = train_oil_df['country'] == 'uk /norway'
nor_uk = train_oil_df['country'] == 'norway /uk'

train_oil_df[uk_nor | nor_uk][
    ['field_name', 'reservoir_unit', 'basin_name', 'country', 'operator_company', 'latitude', 'longitude',
     'onshore_offshore']]

Заменим разные названия.

In [ ]:
train_oil_df.loc[nor_uk, 'country'] = 'uk /norway'

In [ ]:
plot_ct(train_oil_df, 'basin_name', 'Бассейны', figsize=(20, 3))

Месторождения достаточно хорошо разделены по бассейнам.

### Геоанализ

In [ ]:
gdf = gpd.GeoDataFrame(
    train_oil_df_filled,
    geometry=gpd.points_from_xy(train_oil_df_filled.longitude, train_oil_df_filled.latitude),
    crs="EPSG:4326"
)

#### Расположение месторождений

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ocean_110m.plot(ax=ax, **water_col)
gdf.plot(ax=ax, column='onshore_offshore', **field_markers, **field_lgnd)
ax.set_ylabel('Широта')
ax.set_xlabel('Долгота')
ax.grid(False)
fig.suptitle('Расположение месторождений', fontsize=16, fontweight='bold')
plt.show()

#### Подбор масштабов
Для корректных расчётов подходит масштаб карт 1:50000000.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ocean_110m.plot(ax=ax1, **water_col)
gdf.plot(ax=ax1, column='onshore_offshore', **field_markers, **field_lgnd)
ocean_50m.plot(ax=ax2, **water_col)
gdf.plot(ax=ax2, column='onshore_offshore', **field_markers, **field_lgnd)
ax1.set_ylim(5.4, 5.8)
ax1.set_xlim(4.75, 5.5)
ax2.set_ylim(5.4, 5.8)
ax2.set_xlim(4.75, 5.5)
ax1.set_title('Карты масштаба 1:110000000', fontsize=14)
ax2.set_title('Карты масштаба 1:50000000', fontsize=14)
ax1.set_ylabel('Широта')
fig.text(0.5, 0.0, 'Долгота', ha='center', fontsize=12)
fig.suptitle('Сравнение масштабов', fontsize=16, fontweight='bold', y=.98)
plt.tight_layout()
plt.show()

#### Анализ расположения: суша или вода

In [ ]:
gdf['water_pct'] = gdf.apply(lambda row: water_fraction(point_lat=row.latitude,
                                                        point_lon=row.longitude,
                                                        water_shapes=ocean_50m,
                                                        radius_km=2), axis=1)

In [ ]:
gdf.groupby(['onshore_offshore', 'water_pct']).size().reset_index(name='count')

##### Признаки расположения координат на суше и в воде 
Достаточно одного признака: `is_in_water`.

In [ ]:
gdf['is_in_water'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=ocean_50m), axis=1)
gdf.groupby(['onshore_offshore', 'is_in_water']).size().reset_index(name='count')

##### Оффшорные месторождения на суше

In [ ]:
offshore_outlier = (gdf['onshore_offshore'] == 'offshore') & (gdf['water_pct'] == 0)

In [ ]:
gdf[offshore_outlier][['field_name', 'reservoir_unit', 'latitude', 'longitude', 'onshore_offshore']]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ocean_50m.plot(ax=ax, **water_col)
gdf[offshore_outlier].plot(ax=ax, column='onshore_offshore', **field_markers, **field_lgnd)
ax.annotate('Berri Oil Project\n(координаты на берегу)', xy=(49.6167, 26.9833), xytext=(49.38, 26.940), fontsize=10)
ax.scatter(49.6389, 27.1145, color='red', s=10)
ax.annotate('Фактическое расположение', xy=(49.6389, 27.11), xytext=(49.675, 27.11), fontsize=10)
ax.set_ylim(26.75, 27.25)
ax.set_xlim(49.2, 50.25)
ax.set_ylabel('Широта')
ax.set_xlabel('Долгота')
ax.set_title('Berri Oil Project (Hanifa), Саудовская Аравия')
plt.show()

Месторождение Berri Oil Project (Hanifa) в датасете указано, как оффшорное. Согласно информации https://www.gem.wiki/Berri_Oil_Project_(Saudi_Arabia), это месторождение действительно является оффшорным, но имеет другие координаты в море: 27.1145, 49.6389. Присвоим этому месторождению координаты из справочника и пересчитаем `water_pct`.

In [ ]:
berri_hanifa_coords = (27.11, 49.6389)
berri_hanifa_idx = (gdf["field_name"] == "berri") & (gdf["reservoir_unit"] == "hanifa")
gdf.loc[berri_hanifa_idx, 'latitude'] = berri_hanifa_coords[0]
gdf.loc[berri_hanifa_idx, 'longitude'] = berri_hanifa_coords[1]
gdf.loc[berri_hanifa_idx, 'water_pct'] = water_fraction(point_lat=berri_hanifa_coords[0],
                                                        point_lon=berri_hanifa_coords[1],
                                                        water_shapes=ocean_50m,
                                                        radius_km=2)

In [ ]:
gdf[offshore_outlier][['field_name', 'reservoir_unit', 'latitude', 'longitude', 'onshore_offshore', 'water_pct']]

In [ ]:
gdf.groupby(['onshore_offshore', 'water_pct']).size().reset_index(name='count')

##### Береговые месторождения, окружённые водой

In [ ]:
onshore_outliers = (gdf['onshore_offshore'] == 'onshore') & (gdf['water_pct'] > 0.9)

In [ ]:
gdf[onshore_outliers]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ocean_50m.plot(ax=ax1, **water_col)
gdf[onshore_outliers].plot(ax=ax1, column='onshore_offshore', **field_markers, **field_lgnd)
ax1.annotate('Озеро Sabine', xy=(-93.9034, 29.8222), xytext=(-93.78, 29.87),
             fontsize=12, zorder=10)
ax1.set_ylim(29.6, 30.1)
ax1.set_xlim(-94, -93.0)
ax1.set_title('Lower Hackberry\nнаходится в озере Sabine')

ocean_50m.plot(ax=ax2, **water_col)
bunyu_island.plot(ax=ax2, color='salmon', edgecolor='lightslategrey')
gdf[onshore_outliers].plot(ax=ax2, column='onshore_offshore', **field_markers, **field_lgnd)
ax2.annotate('Остров Bunyu', xy=(117.833, 3.5), xytext=(117.55, 3.5), fontsize=12)
ax2.set_ylim(3.2, 3.7)
ax2.set_xlim(117.0, 118)
ax2.set_title('Tarakan Santul Tabul Bunyu\nнаходится на острове Bunyu')
ax1.set_ylabel('Широта')
fig.text(0.5, 0.0, 'Долгота', ha='center', fontsize=12)
fig.suptitle('Береговые месторождения, окружённые водой', fontsize=16, fontweight='bold', y=.98)
plt.tight_layout()
plt.show()

Для таких месторождений следует добавить признаки, указывающие на то, что они находятся в озере или на острове.

In [ ]:
gdf['is_on_island'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=all_islands), axis=1)
gdf['is_in_gulf'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=gulfs), axis=1)
gdf['is_in_strait'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=straits), axis=1)
gdf['is_in_delta'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=deltas), axis=1)
gdf['is_in_bay'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=bays), axis=1)

gdf['is_in_lake'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=all_lakes), axis=1)
gdf['is_in_channel'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=channels), axis=1)
gdf['is_in_sound'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=sounds), axis=1)
gdf['is_in_river'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=rivers), axis=1)
gdf['is_on_isthmus'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=isthmuses), axis=1)
gdf['is_on_coast'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=coasts), axis=1)

Месторождение в озере Sabine отмечено как залив. На странице [Sabine Lake в Wikipedia](https://en.wikipedia.org/wiki/Sabine_Lake) пишут:
> Sabine Lake is a bay on the Gulf coasts of Texas and Louisiana, located approximately 90 miles (140 km) east of Houston and 160 miles (260 km) west of Baton Rouge, adjoining the city of Port Arthur. The lake is formed by the confluence of the Neches and Sabine Rivers and connects to the Gulf of Mexico through Sabine Pass.

По сути, озеро соединено с Мексиканским заливом. Как признак, это тоже подходит, но можно посмотреть ещё упоминания озёр в названиях.

In [ ]:
names_concat = gdf['field_name'] + gdf['reservoir_unit'] + gdf['basin_name']

In [ ]:
print(f"Упоминаний озёр: {names_concat.str.contains("lake").sum()}")

In [ ]:
gdf['ref_lake'] = names_concat.str.contains("lake")

In [ ]:
gdf[onshore_outliers][['field_name', 'latitude', 'longitude', 'is_on_island', 'is_in_gulf', 'ref_lake']]

Посмотрим на разметку новых признаков. Имеют вхождения только признаки для осторовов, заливов, проливов, дельт рек и бухт. Остальные признаки вхождений не имеют.

In [ ]:
geo_features = ['is_on_island', 'is_in_lake', 'is_in_bay', 'is_in_gulf', 'is_in_strait', 'is_in_channel', 'is_in_sound',
                'is_in_river', 'is_on_isthmus', 'is_in_delta', 'is_on_coast']
geo_feature_counts = gdf[geo_features].sum()
available_geo_features = geo_feature_counts[geo_feature_counts > 0]
available_geo_features

##### Месторождения со смешанными значениями

In [ ]:
mixed_on_offshores = gdf['onshore_offshore'] == 'onshore-offshore'

In [ ]:
gdf[mixed_on_offshores][
    ['field_name', 'reservoir_unit', 'basin_name', 'latitude', 'longitude', 'water_pct', 'is_on_island', 'is_in_gulf',
     'is_in_strait',
     'is_in_delta', 'is_in_bay']]

###### Wytch Farm
Wytch Farm находится в южном побережье Соединённого Королевства. Есть корректная отметка о том, что месторождение находится на острове. Остальные месторождения не имеют вхождений по заданным признакам.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ocean_50m.plot(ax=ax, **water_col)

ax.scatter(-2.0278, 50.6672, color='red', s=15)
ax.annotate('Wytch Farm', xy=(-2.0278, 50.6672), xytext=(-1.9, 51.0), fontsize=10)

ax.set_ylim(49, 60)
ax.set_xlim(-7, 2.5)
ax.set_ylabel('Широта')
ax.set_xlabel('Долгота')
ax.set_title('Месторождение Wytch Farm, UK')
plt.show()

##### Tia Juana
Месторождения Tía Juana на ходятся в бассейне Маракайбо.

<img src="../doc/img/Maracaibo_Basin.png" width="300">

Т.к. для этой записи пришлось восстанавливать координаты, стоит отметить, что в справочниках [Tía Juana Oil Field](https://www.gem.wiki/T%C3%ADa_Juana_Oil_Field_(Venezuela)) - это часть проекта "Bolivar Coastal Fields", который включает месторождения:
- Ambrosio Oil and Gas Field
- [Bachaquero Oil Field](https://www.gem.wiki/Bachaquero_Oil_Field_(Venezuela))
- [Barúa Oil Field](https://www.gem.wiki/Bar%C3%BAa_Oil_Field_(Venezuela))
- [Lagunillas Oil Field](https://www.gem.wiki/Lagunillas_Oil_Field_(Venezuela)).

Нас же интересует [Lagunillas Oil Field](https://www.gem.wiki/Lagunillas_Oil_Field_(Venezuela)), т.к. в reservoir unit есть уточнение, что это onshore. Но, что не менее важно, в тексте встерчаются упомнинания onshore/offshore, поэтому можно добавить 2 признака. 

Источники:
- [Tía Juana Oil Field](https://www.gem.wiki/T%C3%ADa_Juana_Oil_Field_(Venezuela))
- https://en.wikipedia.org/wiki/Bolivar_Coastal_Fields
 

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ocean_50m.plot(ax=ax, **water_col)

ax.scatter(-71.0365, 9.8179, color='red', s=15)
ax.annotate('Месторождение\nLagunillas', xy=(-71.0365, 9.8179), xytext=(-71.0, 9.7), fontsize=10)

ax.scatter(-71.7711, 10.3212, color='grey', s=15)
ax.annotate('Месторождение\nTía Juana', xy=(-71.7711, 10.3212), xytext=(-72.3, 10.26), fontsize=10)

ax.set_ylim(9, 11)
ax.set_xlim(-73, -70)
ax.set_ylabel('Широта')
ax.set_xlabel('Долгота')
ax.set_title('Месторождения Bolivar Coastal Fields\nбассейн Маракайбо')
plt.show()

**Cheleken**
Месторождение Челекен находится в восточной части Каспийского моря в окрестностях залива Туркменбаши. Месторождение промаркировано onshore/offshore, т.к. оно расположено частично под сушей и частично под морем.

<img src="../doc/img/Cheleken.png" width="400">

Источник: https://www.researchgate.net/figure/Overview-map-of-the-study-area-along-the-eastern-coast-of-the-Caspian-Sea-Turkmenistan_fig1_328854399

Можно по внешним источникам извлечь сведения о бассейне/месторождении.

In [ ]:
cheleken_idx = gdf['field_name'] == 'cheleken'
gdf[cheleken_idx][['field_name', 'reservoir_unit', 'basin_name', 'water_pct', 'onshore_offshore', 'hydrocarbon_type']]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ocean_50m.plot(ax=ax, **water_col)

ax.scatter(53.1300, 39.5800, color='red', s=15)
ax.annotate('Месторождение\nЧелекен', xy=(53.1300, 39.5800), xytext=(53.1300, 39.5800), fontsize=10)

ax.set_ylim(39.25, 40)
ax.set_xlim(52.5, 54)
ax.set_ylabel('Широта')
ax.set_xlabel('Долгота')
ax.set_title('Месторождения нефтегазового проекта Челекен\nКаспийское море')
plt.show()

**Parentis**
Месторождение Parentis относится к Аквитанскому бассейну в Бискайском заливе. Согласно внешним классификаторам, Аквитанский бассейн является смешанным в смысле onshore/offshore.  

Сами координаты указывают на берег. Возможно, в Field Name имеется в виду не само месторождение Parentis, а провинция (Parentis province), которая действительно залегает под материком и заливом. А само месторождение Parentis находится на берегу. Возможно, придётся корректировать лейбл.

<img src="../doc/img/Parentis.png" width="400">

Источник: https://www.aapg.org/news-and-media/explorer/exploration-adventures-in-frances-aquitaine-basin/

Можно по внешним источникам извлечь сведения о бассейне. 

In [ ]:
parentis_idx = gdf['field_name'] == 'parentis'
gdf[parentis_idx][['field_name', 'reservoir_unit', 'basin_name', 'water_pct', 'onshore_offshore', 'hydrocarbon_type']]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ocean_50m.plot(ax=ax, **water_col)

ax.scatter(-1.0959, 44.3367, color='red', s=15)
ax.annotate('Месторождение\nParentis', xy=(-1.0959, 44.3367), xytext=(-1.0959, 44.38), fontsize=10)

ax.set_ylim(44, 45)
ax.set_xlim(-2, 0)
ax.set_ylabel('Широта')
ax.set_xlabel('Долгота')
ax.set_title('Месторождении Parentis\nберег Бискайского залива')
plt.show()

##### Huntington Beach
Координаты указывают на берег, хотя само месторождение Huntington Beach находится между сушей и океаном.

<img src="../doc/img/LA_Basin.png" width="400">

Источник: https://en.wikipedia.org/wiki/Huntington_Beach_Oil_Field

Можно по внешним источникам извлечь сведения о бассейне.

In [ ]:
huntington_beach_idx = gdf['field_name'] == 'huntington beach'
gdf[huntington_beach_idx][
    ['field_name', 'reservoir_unit', 'basin_name', 'water_pct', 'onshore_offshore', 'hydrocarbon_type']]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ocean_50m.plot(ax=ax, **water_col)

ax.scatter(-118.0335, 33.6998, color='red', s=15)
ax.annotate('Месторождение\nHuntington Beach', xy=(-118.0335, 33.6998), xytext=(-118.0335, 33.6998), fontsize=10)

ax.set_ylim(33.2, 34.2)
ax.set_xlim(-118.5, -117.3)
ax.set_ylabel('Широта')
ax.set_xlabel('Долгота')
ax.set_title('Месторождение Huntington Beach\nберег Тихого Океана')
plt.show()

##### Признаки на основе упоминаний onshore/offshore

In [ ]:
print(f"Упоминаний onshore: {names_concat.str.contains("onshore").sum()}")
print(f"Упоминаний offshore: {names_concat.str.contains("offshore").sum()}")

In [ ]:
gdf['ref_onshore'] = names_concat.str.contains('onshore')
gdf['ref_offshore'] = names_concat.str.contains('offshore')

In [ ]:
gdf.groupby(['onshore_offshore', 'ref_onshore', 'ref_offshore']).size().reset_index(name='count')

Здесь важно: 
- нет `offshore` с упоминаниями `onshore`;
- нет `onshore` с упоминаниями `offshore`;
- есть `onshore-offshore` запись с упоминанием `onshore`.

Получилось промаркировать месторождение Tia Juana признаком, который может помочь определению локации.

In [ ]:
gdf[mixed_on_offshores][
    ['field_name', 'reservoir_unit', 'basin_name', 'ref_onshore']]

##### Признаки на основе обучающих данных о бассейне

In [ ]:
basin_location = pd.crosstab(gdf['basin_name'], gdf['onshore_offshore']).drop('unknown', errors='ignore').reset_index()

loc_mask = ((basin_location['onshore'] > 0) &
            (basin_location['offshore'] > 0))

basin_location.loc[loc_mask, 'onshore-offshore-calc'] = (
        basin_location.loc[loc_mask, 'onshore'] +
        basin_location.loc[loc_mask, 'offshore'] +
        basin_location.loc[loc_mask, 'onshore-offshore'])

basin_location['onshore-offshore-calc'] = basin_location['onshore-offshore-calc'].fillna(0).astype(int)

loc_mask_2 = ((basin_location['onshore'] == 0) &
              (basin_location['offshore'] == 0) &
              (basin_location['onshore-offshore'] > 0))

basin_location.loc[loc_mask_2, 'onshore-offshore-calc'] = basin_location.loc[loc_mask_2, 'onshore-offshore']

In [ ]:
inferred_location_map = construct_basin_location_mapper(basin_location)

In [ ]:
gdf['basin_location'] = gdf['basin_name'].map(inferred_location_map).fillna("unknown")

In [ ]:
gdf.groupby(['onshore_offshore', 'basin_location']).size().reset_index(name='count')

##### Признаки на основе внешних данных о бассейне
Рассмотрим данные о бассейне, которые мы загрузили в ноутбуке [load_basins_data](load_basins_data.ipynb).

In [ ]:
basins_mapped = pd.read_csv("../data/basins_mapped.csv")
basins_mapped = basins_mapped.drop(basins_mapped[basins_mapped['basin_name'] == 'unknown'].index).reset_index(
    drop=True).to_dict("records")
external_location_map = {b['basin_name']: b['location'] for b in basins_mapped}

In [ ]:
basin_location_map_combined = construct_basin_location_mapper_with_external(basin_location, external_location_map)

In [ ]:
gdf['basin_location_ext'] = gdf['basin_name'].map(basin_location_map_combined).fillna('unknown')

In [ ]:
gdf.groupby(['onshore_offshore', 'basin_location']).size().reset_index(name='count')

In [ ]:
gdf.groupby(['onshore_offshore', 'basin_location_ext']).size().reset_index(name='count')

Разметки расходятся. Оставим обе.

### Анализ распределений 

#### Тектонический режим

In [ ]:
tectonic_regime = split_strings(gdf, 'tectonic_regime')
tectonic_regime_df = pd.DataFrame({"tectonic_regime": tectonic_regime, "onshore_offshore": gdf['onshore_offshore']})
tectonic_regime_df = tectonic_regime_df.explode("tectonic_regime").reset_index(drop=True)

In [ ]:
plot_ct(tectonic_regime_df, 'tectonic_regime', 'Тектонический режим месторождений')

Compression, erosion, evaporative, extension более характерны для береговых месторождений. Gravity и inversion - для оффшорных.

#### Структурная обстановка

In [ ]:
structural_setting = split_strings(gdf, 'structural_setting')
structural_setting_df = pd.DataFrame(
    {"structural_setting": structural_setting, "onshore_offshore": gdf['onshore_offshore']})
structural_setting_df = structural_setting_df.explode("structural_setting").reset_index(drop=True)

In [ ]:
plot_ct(structural_setting_df, 'structural_setting', 'Структурная обстановка месторождений')

Foreland, intracratonic, passive margin, sub-thrust и thrust более характерны для береговых месторождений. А rift и inversion - для оффшорных.

#### Геологический период месторождений

In [ ]:
plot_ct(gdf, 'reservoir_period', 'Литология месторождений')

Оффшорные месторождения представлены только меловым, мелово-палеогеновым, юрским, неогеновым, палеогеновым, палеогеново-неогеновым, палеозойским, пермским, триасово-юрским периодами. Среди месторождений юрского периода преобладают оффшорные.   

#### Литология месторождений

In [ ]:
plot_ct(gdf, 'lithology', 'Литология месторождений')

Оффшорные месторождения представлены преимущественно известняками и песчаником.

#### Типы углеводородов

In [ ]:
plot_ct(gdf, 'hydrocarbon_type', 'Виды углеводорода месторождений')

Нефтяные месторождения преимущественно располагаются на берегу.

#### Состояние месторождения

In [ ]:
plot_ct(gdf, 'reservoir_status', 'Состояние месторождений')

Статусы "снижение добычи", "поздняя стадия разработки" и "почти исчерпано", "восстановление" преимущественно представлены береговыми месторождениями.

#### Страны

In [ ]:
plot_ct(gdf, 'country', 'Распределение месторождений по странам', figsize=(18, 3))

Месторождения достаточно неплохо распределены и по странам.

#### Регионы

In [ ]:
plot_ct(gdf, 'region', 'Распределение месторождений по регионам', figsize=(18, 3))

Можно заметить, что в Северной Америке преобладают береговые месторождения, а в Европе - оффшорные.

#### Численные характеристики месторождений

In [ ]:
numeric_cols = ['depth', 'thickness_gross_average_ft', 'thickness_net_pay_average_ft', 'porosity', 'permeability']
gdf_numeric = gdf[numeric_cols + ['onshore_offshore']].copy()
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
gdf_scaled = gdf_numeric.copy()
gdf_scaled[numeric_cols] = scaler.fit_transform(gdf_numeric[numeric_cols])

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(3 * len(numeric_cols), 3))
fig.suptitle('Плотность вероятности для числовых признаков\nпо месторасположению месторождений',
             fontsize=12, fontweight='bold', y=0.98)

for i, col in enumerate(numeric_cols):
    for loc in ['onshore', 'offshore', 'onshore-offshore']:
        subset = gdf_numeric[gdf_numeric['onshore_offshore'] == loc][col]
        axes[i].hist(subset, bins=20, alpha=0.5, label=loc, density=True)
    for ax in axes:
        ax.tick_params(left=False, labelleft=False)
    axes[i].set_title(col)
    axes[i].legend()

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(3 * len(numeric_cols), 3))
fig.suptitle('Плотность вероятности для нормализованных числовых признаков\nпо месторасположению месторождений',
             fontsize=12, fontweight='bold', y=0.98)

for i, col in enumerate(numeric_cols):
    for loc in ['onshore', 'offshore', 'onshore-offshore']:
        subset = gdf_scaled[gdf_scaled['onshore_offshore'] == loc][col]
        axes[i].hist(subset, bins=20, alpha=0.5, label=loc, density=True)
    for ax in axes:
        ax.tick_params(left=False, labelleft=False)
    axes[i].set_title(col)
    axes[i].legend()

plt.tight_layout()

Смешанные месторождения отличаются по плотности распределения вероятности.

In [ ]:
corr_matrix = gdf_numeric.groupby('onshore_offshore')[numeric_cols].corr()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, loc in enumerate(['onshore', 'offshore', 'onshore-offshore']):
    sns.heatmap(corr_matrix.loc[loc], ax=axes[i], annot=True, cmap='coolwarm',
                cbar=(i == 2), vmin=-1, vmax=1)
    axes[i].set_title(loc)
plt.tight_layout()

### Test


#### Проверка содержания категориальных признаков

In [18]:
test_oil_df.columns.tolist()

['field_name',
 'reservoir_unit',
 'country',
 'region',
 'basin_name',
 'tectonic_regime',
 'latitude',
 'longitude',
 'operator_company',
 'hydrocarbon_type',
 'reservoir_status',
 'structural_setting',
 'depth',
 'reservoir_period',
 'lithology',
 'thickness_gross_average_ft',
 'thickness_net_pay_average_ft',
 'porosity',
 'permeability']

In [14]:
test_oil_df.shape

(133, 19)

In [20]:
test_id = test_oil_df['field_name'] + test_oil_df['reservoir_unit']
train_id = train_oil_df['field_name'] + train_oil_df['reservoir_unit']
test_id.name = 'id'
train_id.name = 'id'

print(f"field_name + reservoir_unit: {len(cat_diff(train_id.to_frame(), test_id.to_frame(), 'id'))}")
print(f"field_name: {len(cat_diff(train_oil_df, test_oil_df, 'field_name'))}")
print(f"basin_name: {len(cat_diff(train_oil_df, test_oil_df, 'basin_name'))}")
print(f"reservoir_unit: {len(cat_diff(train_oil_df, test_oil_df, 'reservoir_unit'))}")
print(f"lithology: {len(cat_diff(train_oil_df, test_oil_df, 'lithology'))}")
print(f"tectonic_regime: {len(cat_diff(train_oil_df, test_oil_df, 'tectonic_regime'))}")
print(f"structural_setting: {len(cat_diff(train_oil_df, test_oil_df, 'structural_setting'))}")
print(f"reservoir_status: {len(cat_diff(train_oil_df, test_oil_df, 'reservoir_status'))}")
print(f"region: {len(cat_diff(train_oil_df, test_oil_df, 'region'))}")
print(f"country: {len(cat_diff(train_oil_df, test_oil_df, 'country'))}")
print(f"operator_company: {len(cat_diff(train_oil_df, test_oil_df, 'operator_company'))}")
print(f"hydrocarbon_type: {len(cat_diff(train_oil_df, test_oil_df, 'hydrocarbon_type'))}")

field_name + reservoir_unit: 133
field_name: 103
basin_name: 15
reservoir_unit: 95
lithology: 4
tectonic_regime: 6
structural_setting: 10
reservoir_status: 0
region: 0
country: 6
operator_company: 42
hydrocarbon_type: 2


#### Проверка пустых значений

In [23]:
missing_val_cnts_int_test = test_oil_df.isna().sum()
missing_val_cnts_int_test[missing_val_cnts_int_test > 0]

country       13
region        16
basin_name     8
latitude      13
longitude     16
dtype: int64

##### Координаты меторождения Djeitun
На сайте https://sst-ts.com/wellhead-platform-zhdanov-a/ указано, что Джейтун входит в контрактный район Челекен. Возьмём координаты Челекен. Оператор Dragon Oil.

In [26]:
pd.concat([
    test_oil_df[test_oil_df['field_name'] == 'djeitun'][['field_name', 'reservoir_unit', 'latitude', 'longitude', 'operator_company']],
    train_oil_df[train_oil_df['field_name'] == 'cheleken'][['field_name', 'reservoir_unit', 'latitude', 'longitude', 'operator_company']]
])

,field_name,reservoir_unit,latitude,longitude,operator_company
35,djeitun,red series,39.58,NaN,dragon oil
134,cheleken,red series,39.58,53.13,turkmenneft


##### Заполнение координат

In [10]:
test_oil_df[test_oil_df['field_name'] == 'belayim marine']

,field_name,reservoir_unit,country,region,basin_name,tectonic_regime,latitude,longitude,operator_company,hydrocarbon_type,reservoir_status,structural_setting,depth,reservoir_period,lithology,thickness_gross_average_ft,thickness_net_pay_average_ft,porosity,permeability
13,belayim marine,rudeis-kareem,NaN,NaN,gulf of suez,extension/evaporite,NaN,NaN,petrobel,oil,declining production,sub-salt/rift,7090,neogene,sandstone,1300.0,500.0,18.0,400.0
